# Test the Gene Annotation Pipeline

This notebook validates the input reader, local human annotation adapters, and the complete mouse pipeline.

## Step 1 — Configure paths and imports

In [8]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.input_output import read_gene_csv
from src.annotate_local_files import annotate_hgnc, annotate_ligand_receptor
from src.pipeline import run_pipeline

input_example_csv = project_root / "data" / "example" / "gene_list_MC38OT1_pos.csv"
annotation_dir = project_root / "data" / "annotation"
output_dir = project_root / "results" / "notebook_test"

print("Input:", input_example_csv)
print("Annotations:", annotation_dir)
print("Output:", output_dir)

Input: /mnt/c/Users/aweyr/OneDrive/Documents/Cheng-Lab/Coculture-Project/cancer-immunology-gene-annotation/data/example/gene_list_MC38OT1_pos.csv
Annotations: /mnt/c/Users/aweyr/OneDrive/Documents/Cheng-Lab/Coculture-Project/cancer-immunology-gene-annotation/data/annotation
Output: /mnt/c/Users/aweyr/OneDrive/Documents/Cheng-Lab/Coculture-Project/cancer-immunology-gene-annotation/results/notebook_test


## Step 2 — Inspect the mouse input CSV

In [9]:
input_df = pd.read_csv(input_example_csv)

display(input_df.head())
print("Columns:", list(input_df.columns))

,Unnamed: 0,0
0,0,Tradd
1,1,Tnfrsf1a
2,2,Ado
3,3,Zfx
4,4,Fadd


Columns: ['Unnamed: 0', '0']


## Step 3 — Test gene-list cleaning

In [10]:
mouse_genes = read_gene_csv(input_example_csv)

assert len(mouse_genes) == 50
assert len(mouse_genes) == len(set(mouse_genes))
assert all(gene.strip() for gene in mouse_genes)

print(f"Read {len(mouse_genes)} unique mouse genes.")

Read 50 unique mouse genes.


## Step 4 — Test the local human HGNC and ligand–receptor adapters

These are **human-only local resources**. A mouse list cannot be tested against them offline unless mouse genes have already been converted to human orthologs. Therefore, this step uses a small human gene-symbol list directly.

In [11]:
human_test_genes = ["TRADD", "TNFRSF1A", "FADD", "CASP8", "IFNGR1"]

hgnc_path = annotation_dir / "gene annotation.txt"
lr_path = annotation_dir / "ligand_receptor.csv"

assert hgnc_path.exists(), f"Missing HGNC file: {hgnc_path}"
assert lr_path.exists(), f"Missing ligand-receptor file: {lr_path}"

hgnc_data = pd.read_csv(hgnc_path, sep="\t", low_memory=False)
lr_data = pd.read_csv(lr_path, low_memory=False)

hgnc_test = annotate_hgnc(human_test_genes, hgnc_data)
lr_test = annotate_ligand_receptor(human_test_genes, lr_data)

assert not hgnc_test.empty
assert "Approved symbol" in hgnc_test.columns
assert set(human_test_genes).issubset(set(hgnc_test["input_gene"]))

print("HGNC rows:", len(hgnc_test))
print("Ligand-receptor rows:", len(lr_test))

HGNC rows: 24
Ligand-receptor rows: 355


## Step 5 — Run a local-only mouse pipeline test

In [12]:
local_tables = run_pipeline(
    input_csv = input_example_csv,
    organism = "mouse",
    output_dir = output_dir / "local_only",
    gene_column = None,
    annotation_dir = annotation_dir,
    use_remote = False,
)

display(pd.DataFrame([
    {"table": name, "rows": len(table), "columns": len(table.columns)}
    for name, table in local_tables.items()
]))

display(local_tables["run_errors"])

,table,rows,columns
0,input_genes,50,1
1,hgnc,0,0
2,ligand_receptor,0,0
3,run_errors,2,2


,step,error
0,hgnc,Mouse HGNC annotation requires human orthologs...
1,ligand_receptor,Mouse ligand-receptor annotation requires huma...


The empty `hgnc` and `ligand_receptor` tables above are expected for a local-only **mouse** run because human orthologs require the remote orthology step. The error table now explains this explicitly.

## Step 6 — Run the complete remote-enabled mouse pipeline

In [13]:
remote_tables = run_pipeline(
    input_csv = input_example_csv,
    organism = "mouse",
    output_dir = output_dir / "remote",
    gene_column = None,
    annotation_dir = annotation_dir,
    use_remote = True
)

summary = pd.DataFrame([
    {"table": name, "rows": len(table), "columns": len(table.columns)}
    for name, table in remote_tables.items()
])
    
display(summary)

2 input query terms found no hit:	['AU022751', 'Fam26f']


,table,rows,columns
0,input_genes,50,1
1,gprofiler_ids,51,8
2,gprofiler_enrichment,79,16
3,mygene,50,22
4,mouse_to_human_orthologs,51,9
5,hgnc,334,17
6,ligand_receptor,896,10
7,run_errors,0,2


## Step 7 — Inspect returned annotation tables

In [14]:
for name in [
    "gprofiler_ids",
    "gprofiler_enrichment",
    "mygene",
    "mouse_to_human_orthologs",
    "hgnc",
    "ligand_receptor"
]:
    print(f"\n{name}: {len(remote_tables[name])} rows")
    display(remote_tables[name].head())


gprofiler_ids: 51 rows


,incoming,converted,n_incoming,n_converted,name,description,namespaces,query
0,Tradd,ENSMUSG00000031887,1,1,Tradd,TNFRSF1A-associated via death domain [Source:M...,"ENTREZGENE,MGI,UNIPROT_GN,WIKIGENE",query_1
1,Tnfrsf1a,ENSMUSG00000030341,2,1,Tnfrsf1a,"tumor necrosis factor receptor superfamily, me...","ENTREZGENE,MGI,UNIPROT_GN,WIKIGENE",query_1
2,Ado,ENSMUSG00000057134,3,1,Ado,2-aminoethanethiol dioxygenase [Source:MGI Sym...,"ENTREZGENE,MGI,UNIPROT_GN,WIKIGENE",query_1
3,Zfx,ENSMUSG00000079509,4,1,Zfx,zinc finger protein X-linked [Source:MGI Symbo...,"ENTREZGENE,MGI,UNIPROT_GN,WIKIGENE",query_1
4,Fadd,ENSMUSG00000031077,5,1,Fadd,Fas associated via death domain [Source:MGI Sy...,"ENTREZGENE,MGI,UNIPROT_GN,WIKIGENE",query_1



gprofiler_enrichment: 79 rows


,source,native,name,p_value,significant,description,term_size,query_size,intersection_size,effective_domain_size,precision,recall,query,parents,intersections,evidences
0,KEGG,KEGG:04217,Necroptosis,8.742038e-08,True,Necroptosis,176,22,8,10539,0.363636,0.045455,query_1,[KEGG:00000],"[Tradd, Tnfrsf1a, Fadd, Jak1, Casp8, Sqstm1, B...","[[KEGG], [KEGG], [KEGG], [KEGG], [KEGG], [KEGG..."
1,KEGG,KEGG:05164,Influenza A,2.549475e-06,True,Influenza A,173,22,7,10539,0.318182,0.040462,query_1,[KEGG:00000],"[Tradd, Tnfrsf1a, Fadd, Jak1, Casp8, Bid, Ifngr1]","[[KEGG], [KEGG], [KEGG], [KEGG], [KEGG], [KEGG..."
2,KEGG,KEGG:05152,Tuberculosis,3.352008e-06,True,Tuberculosis,180,22,7,10539,0.318182,0.038889,query_1,[KEGG:00000],"[Tradd, Tnfrsf1a, Fadd, Jak1, Casp8, Bid, Ifngr1]","[[KEGG], [KEGG], [KEGG], [KEGG], [KEGG], [KEGG..."
3,KEGG,KEGG:05168,Herpes simplex virus 1 infection,8.467624e-06,True,Herpes simplex virus 1 infection,206,22,7,10539,0.318182,0.033981,query_1,[KEGG:00000],"[Tradd, Tnfrsf1a, Fadd, Jak1, Casp8, Bid, Ifngr1]","[[KEGG], [KEGG], [KEGG], [KEGG], [KEGG], [KEGG..."
4,WP,WP:WP166,Apoptosis modulation by HSP70,1.174772e-05,True,Apoptosis modulation by HSP70,17,18,4,4801,0.222222,0.235294,query_1,[WP:000000],"[Tnfrsf1a, Fadd, Casp8, Bid]","[[WP], [WP], [WP], [WP]]"



mygene: 50 rows


...

,mouse_gene,human_gene,human_ensembl_gene_id,n_incoming,n_converted,n_result,human_gene_name,human_gene_description,namespaces
0,Tradd,ENSMUSG00000031887,ENSG00000102871,1,1,1,TRADD,TNFRSF1A associated via death domain [Source:H...,"ENTREZGENE,MGI,UNIPROT_GN,WIKIGENE"
1,Tnfrsf1a,ENSMUSG00000030341,ENSG00000067182,2,1,1,TNFRSF1A,TNF receptor superfamily member 1A [Source:HGN...,"ENTREZGENE,MGI,UNIPROT_GN,WIKIGENE"
2,Ado,ENSMUSG00000057134,ENSG00000181915,3,1,1,ADO,2-aminoethanethiol dioxygenase [Source:HGNC Sy...,"ENTREZGENE,MGI,UNIPROT_GN,WIKIGENE"
3,Zfx,ENSMUSG00000079509,ENSG00000067646,4,1,1,ZFY,zinc finger protein Y-linked [Source:HGNC Symb...,"ENTREZGENE,MGI,UNIPROT_GN,WIKIGENE"
4,Fadd,ENSMUSG00000031077,ENSG00000168040,5,1,1,FADD,Fas associated via death domain [Source:HGNC S...,"ENTREZGENE,MGI,UNIPROT_GN,WIKIGENE"



hgnc: 334 rows


,input_gene,HGNC ID,Status,Approved symbol,Approved name,Alias symbol,Alias name,Previous symbol,Previous name,Chromosome,Chromosome location,Locus group,HGNC family name,HGNC family ID,UniProt accession,Ensembl gene ID,RefSeq accession
0,TRADD,HGNC:12030,Approved,TRADD,TNFRSF1A associated via death domain,Hs.89862,NaN,NaN,TNFRSF1A-associated via death domain,16,16q22.1,protein-coding gene,NaN,NaN,Q15628,ENSG00000102871,NM_001323552
1,TNFRSF1A,HGNC:11916,Approved,TNFRSF1A,TNF receptor superfamily member 1A,TNF-R,NaN,TNFR1,"tumor necrosis factor receptor superfamily, me...",12,12p13.31,protein-coding gene,NaN,NaN,P19438,ENSG00000067182,NM_001065
2,TNFRSF1A,HGNC:11916,Approved,TNFRSF1A,TNF receptor superfamily member 1A,TNFAR,NaN,TNFR1,"tumor necrosis factor receptor superfamily, me...",12,12p13.31,protein-coding gene,NaN,NaN,P19438,ENSG00000067182,NM_001065
3,TNFRSF1A,HGNC:11916,Approved,TNFRSF1A,TNF receptor superfamily member 1A,TNFR60,NaN,TNFR1,"tumor necrosis factor receptor superfamily, me...",12,12p13.31,protein-coding gene,NaN,NaN,P19438,ENSG00000067182,NM_001065
4,TNFRSF1A,HGNC:11916,Approved,TNFRSF1A,TNF receptor superfamily member 1A,TNF-R-I,NaN,TNFR1,"tumor necrosis factor receptor superfamily, me...",12,12p13.31,protein-coding gene,NaN,NaN,P19438,ENSG00000067182,NM_001065



ligand_receptor: 896 rows


,input_gene,lr_role,source (gene ID),target (gene ID),LR_pair,is_stimulation,is_inhibition,interaction,evidence,data source
0,CASP8,source,CASP8,FAS,CASP8_FAS,NaN,NaN,Cell adhesion,PMID: 19118384;,Celllinker
1,CORIN,source,CORIN,KIRREL,CORIN_KIRREL,NaN,NaN,Cell adhesion,PMID: 32589946;,Celllinker
2,CORIN,source,CORIN,LRRC4C,CORIN_LRRC4C,NaN,NaN,Cell adhesion,PMID: 32589946;,Celllinker
3,IFNGR1,source,IFNGR1,FCAR,IFNGR1_FCAR,NaN,NaN,Cell adhesion,PMID: 32822567;,Celllinker
4,JAK1,source,JAK1,EGFR,JAK1_EGFR,NaN,NaN,Cell adhesion,PMID: 23045285;,Celllinker
